Load and Inspect the datasets

In [85]:
#!pip install pandas scikit-learn numpy matplotlib ipaddress SMOTE imblearn


In [1]:
import pandas as pd

fraud_df = pd.read_csv("../data/Fraud_Data.csv")
ip_df = pd.read_csv("../data/IpAddress_to_Country.csv")
creditcard_df = pd.read_csv("../data/creditcard.csv")

# Basic inspection
print(fraud_df.info())
print(ip_df.info())
print(creditcard_df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   user_id         151112 non-null  int64  
 1   signup_time     151112 non-null  object 
 2   purchase_time   151112 non-null  object 
 3   purchase_value  151112 non-null  int64  
 4   device_id       151112 non-null  object 
 5   source          151112 non-null  object 
 6   browser         151112 non-null  object 
 7   sex             151112 non-null  object 
 8   age             151112 non-null  int64  
 9   ip_address      151112 non-null  float64
 10  class           151112 non-null  int64  
dtypes: float64(1), int64(4), object(6)
memory usage: 12.7+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138846 entries, 0 to 138845
Data columns (total 3 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   

Handle Missing Values and Duplicates

In [2]:
# Drop duplicates
fraud_df.drop_duplicates(inplace=True)
creditcard_df.drop_duplicates(inplace=True)

# Fill missing categorical columns
for col in ['sex', 'browser', 'source']:
    if col in fraud_df.columns:
        fraud_df.fillna({'sex': 'Unknown', 'browser': 'Unknown', 'source': 'Unknown'}, inplace=True)

# Drop rows only if critical features are missing
fraud_df.dropna(subset=['signup_time', 'purchase_time', 'purchase_value', 'class'], inplace=True)
creditcard_df.dropna(inplace=True)


In [3]:
print(fraud_df.columns.tolist())


['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id', 'source', 'browser', 'sex', 'age', 'ip_address', 'class']


Fix datatypes

In [4]:
# Convert timestamps to datetime
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])


Feature Engineering-Fraud Data

In [5]:
from datetime import datetime

# Time features
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek
fraud_df['time_since_signup'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds() / 3600

fraud_df.drop(columns=['transaction_count'], errors='ignore', inplace=True)

# Transaction frequency
transaction_counts = fraud_df.groupby('user_id')['purchase_time'].count().rename('transaction_count')
fraud_df = fraud_df.merge(transaction_counts, on='user_id', how='left')


Merge API address to country

In [6]:
print(fraud_df['ip_address'].head(10))
print(fraud_df['ip_address'].dtype)


0    7.327584e+08
1    3.503114e+08
2    2.621474e+09
3    3.840542e+09
4    4.155831e+08
5    2.809315e+09
6    3.987484e+09
7    1.692459e+09
8    3.719094e+09
9    3.416747e+08
Name: ip_address, dtype: float64
float64


In [7]:
import ipaddress
import pandas as pd

# Step 1: Ensure ip_address is a clean integer string
fraud_df['ip_address'] = fraud_df['ip_address'].astype(str)
fraud_df['ip_address'] = fraud_df['ip_address'].str.extract(r'(\d+)')[0]

# Step 2: Convert to integer
fraud_df['ip_int'] = fraud_df['ip_address'].astype(int)

# Step 3: Make sure the IP mapping ranges are integers
ip_df['lower_bound_ip_address'] = ip_df['lower_bound_ip_address'].astype(int)
ip_df['upper_bound_ip_address'] = ip_df['upper_bound_ip_address'].astype(int)

# Step 4: Define country lookup function
def get_country(ip):
    match = ip_df[(ip_df['lower_bound_ip_address'] <= ip) & (ip_df['upper_bound_ip_address'] >= ip)]
    return match['country'].values[0] if not match.empty else "Unknown"

# Step 5: Apply the lookup
fraud_df['country'] = fraud_df['ip_int'].apply(get_country)

# Optional: Preview the result
fraud_df[['ip_address', 'ip_int', 'country']].head()



,ip_address,ip_int,country
0,732758368,732758368,Japan
1,350311387,350311387,United States
2,2621473820,2621473820,United States
3,3840542443,3840542443,Unknown
4,415583117,415583117,United States


Encode catagorical Features

In [25]:
# Only one-hot encode columns that are present
categorical_cols = [col for col in ['source', 'browser', 'sex', 'country'] if col in fraud_df.columns]
fraud_df = pd.get_dummies(fraud_df, columns=categorical_cols, drop_first=True)


In [27]:
print(fraud_df.columns.tolist())


['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id', 'age', 'ip_address', 'class', 'hour_of_day', 'day_of_week', 'time_since_signup', 'transaction_count', 'ip_int', 'source_Direct', 'source_SEO', 'browser_FireFox', 'browser_IE', 'browser_Opera', 'browser_Safari', 'sex_M', 'country_Albania', 'country_Algeria', 'country_Angola', 'country_Antigua and Barbuda', 'country_Argentina', 'country_Armenia', 'country_Australia', 'country_Austria', 'country_Azerbaijan', 'country_Bahamas', 'country_Bahrain', 'country_Bangladesh', 'country_Barbados', 'country_Belarus', 'country_Belgium', 'country_Belize', 'country_Benin', 'country_Bermuda', 'country_Bhutan', 'country_Bolivia', 'country_Bonaire; Sint Eustatius; Saba', 'country_Bosnia and Herzegowina', 'country_Botswana', 'country_Brazil', 'country_British Indian Ocean Territory', 'country_Brunei Darussalam', 'country_Bulgaria', 'country_Burkina Faso', 'country_Burundi', 'country_Cambodia', 'country_Cameroon', 'country_Canada', 'c

Normaize and Scale

In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

fraud_numeric_cols = ['purchase_value', 'age', 'hour_of_day', 'day_of_week', 'time_since_signup', 'transaction_count']
fraud_df[fraud_numeric_cols] = scaler.fit_transform(fraud_df[fraud_numeric_cols])

# For creditcard.csv
creditcard_df['Amount'] = scaler.fit_transform(creditcard_df[['Amount']])


Handle Class Imbalance(SMOTE)

In [41]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Fraud_Data
X_fraud = fraud_df.drop(columns=['class', 'user_id', 'device_id', 'signup_time', 'purchase_time', 'ip_address', 'ip_int'])
y_fraud = fraud_df['class']

X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = train_test_split(X_fraud, y_fraud, test_size=0.2, stratify=y_fraud, random_state=42)

sm = SMOTE(random_state=42)
X_train_fraud_res, y_train_fraud_res = sm.fit_resample(X_train_fraud, y_train_fraud)

# Creditcard
X_cc = creditcard_df.drop(columns=['Class'])
y_cc = creditcard_df['Class']

X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split(X_cc, y_cc, test_size=0.2, stratify=y_cc, random_state=42)

X_train_cc_res, y_train_cc_res = sm.fit_resample(X_train_cc, y_train_cc)


#Why SMOTE Was Used?

In fraud detection, the dataset is typically highly imbalanced — the number of legitimate (non-fraud) transactions vastly outnumbers the fraudulent ones. This imbalance causes machine learning models to become biased toward predicting the majority class, often leading to:
High overall accuracy but poor fraud recall (i.e., many fraud cases go undetected)
Low sensitivity to the minority class (fraud)

To address this, we used SMOTE (Synthetic Minority Over-sampling Technique), which:

Generates synthetic examples of the minority class (fraudulent cases) by interpolating between existing samples, rather than duplicating them
Helps the model learn more generalized fraud patterns without overfitting
Balances the dataset before training, improving model sensitivity (recall) and AUC
Compared to random oversampling or undersampling:
SMOTE is more effective because it introduces diversity in synthetic data
It avoids loss of majority class information, which happens in undersampling

Thus, SMOTE helps build a more fair and robust classifier in imbalanced fraud detection settings.



In [ ]:
#Save the processed data
import os

os.makedirs("../data/processed", exist_ok=True)

fraud_df.to_csv("../data/processed/fraud_data_processed.csv", index=False)
